# Classical ML Models Performance Comparison

Compare all trained classical machine learning models (SVM, k-NN, Decision Tree, Random Forest, Logistic Regression) on the test dataset.

In [ ]:
# Check if running in Google Colab and set up environment cleanly
import os
import shutil
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    # 1. Force change directory back to the root '/content'
    os.chdir('/content')
    
    # 2. Clean up any accidental nested clone directories to save space and resolve path confusion
    nested_path = Path('/content/Bird-Species-Classification-ML/Bird-Species-Classification-ML')
    if nested_path.exists():
        print("Cleaning up accidental nested git clone folders...")
        shutil.rmtree(nested_path, ignore_errors=True)
        
    # 3. Clone repository if it doesn't exist under /content
    if not os.path.exists('Bird-Species-Classification-ML'):
        print("Cloning Bird-Species-Classification-ML repository...")
        !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
        
    # 4. Change working directory to '/content/Bird-Species-Classification-ML'
    %cd /content/Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
# Load pre-computed features and labels
import os
import numpy as np
import joblib
from pathlib import Path

DATA_DIR = Path("./processed_data")
npz_file = DATA_DIR / "combined_hog_color_lbp.npz"

if not npz_file.exists():
    print("Feature file not found. Automatically triggering feature extraction pipeline...")
    import sys
    sys.path.append(str(Path(".").resolve()))
    from src.feature_extraction import build_feature_matrices
    build_feature_matrices()

data = np.load(npz_file)
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

label_mapping = joblib.load(DATA_DIR / "label_mapping.pkl")
class_names = [k.split('.')[-1].replace('_', ' ') for k in sorted(label_mapping, key=label_mapping.get)]

print(f"Loaded feature dataset: {npz_file.name}")
print(f"  Training set  : {X_train.shape}")
print(f"  Validation set: {X_val.shape}")
print(f"  Testing set   : {X_test.shape}")
print(f"  Classes ({len(class_names)}): {class_names[:5]}...")

## 1. Load All Trained Models

In [ ]:
import joblib
from pathlib import Path

MODELS_DIR = Path("./models")
models = {}

model_files = {
    "SVM": "svm_model.pkl",
    "k-NN": "knn_model.pkl",
    "Decision Tree": "decision_tree_model.pkl",
    "Random Forest": "random_forest_model.pkl",
    "Logistic Regression": "logistic_regression_model.pkl"
}

for model_name, filename in model_files.items():
    path = MODELS_DIR / filename
    if path.exists():
        models[model_name] = joblib.load(path)
        print(f"Loaded model: {model_name}")
    else:
        print(f"[Warning] Model file missing: {filename}")

print(f"\nTotal loaded models for evaluation: {len(models)}")

## 2. Evaluate All Models & Build Leaderboard

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics_list = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    metrics_list.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2)
    })

leaderboard_df = pd.DataFrame(metrics_list).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)
print("=== CLASSICAL ML MODEL LEADERBOARD ===")
display(leaderboard_df)

## 3. Visualize Model Performance Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
ax = sns.barplot(x="Model", y="Accuracy (%)", data=leaderboard_df, palette="viridis")

for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='bottom', fontsize=10, xytext=(0, 5), 
                textcoords='offset points')

plt.title("Bird Species Classification - Model Test Accuracy Comparison", fontsize=14)
plt.ylim(0, 100)
plt.ylabel("Test Accuracy (%)")
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()